# 06 · Final Model, Evaluation and Export
Refit the tuned models on more data, choose the final model and decision threshold, evaluate **once** on the untouched test set, and export the bundle used by the web app.

**Reads:** `data/processed/train.parquet`, `data/processed/test.parquet`, `model/feature_config.joblib`, `model/best_params.json`, `data/raw/airlines.csv`, `data/raw/airports.csv`  
**Writes:** `model/flight_delay_bundle.joblib`, `outputs/tuned_model_comparison.csv`

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix, classification_report, f1_score)
from sklearn.calibration import calibration_curve
import sklearn

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


In [ ]:
cfg = joblib.load(MODEL_DIR / "feature_config.joblib")
FEATURES, TARGET = cfg["features"], cfg["target"]
train = pd.read_parquet(DATA_PROC / "train.parquet")
print("train:", train.shape, "| features:", len(FEATURES))
test = pd.read_parquet(DATA_PROC / "test.parquet")
prior, rate_tables = cfg["prior"], cfg["rate_tables"]
with open(MODEL_DIR / "best_params.json") as f:
    best_params = json.load(f)
airlines = pd.read_csv(DATA_RAW / "airlines.csv")
airports = pd.read_csv(DATA_RAW / "airports.csv")
airline_names = airlines.set_index("IATA_CODE").AIRLINE
print("test:", test.shape)

### Helpers

In [ ]:
def best_threshold(y, p):
    """Probability threshold that maximises F1."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]

def get_model(name):
    if name == "Logistic Regression":
        return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
    if name == "Random Forest":
        return Pipeline([("clf", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                                                        n_jobs=-1, random_state=SEED))])
    return Pipeline([("clf", HistGradientBoostingClassifier(random_state=SEED))])

MODELS = ["Logistic Regression", "Random Forest", "HistGradientBoosting"]


## Refit the tuned models on more data, choose the final model and the decision threshold

In [ ]:
final_train = train.sample(min(500_000, len(train)), random_state=SEED + 1)
X_fit, X_val, y_fit, y_val = train_test_split(final_train[FEATURES], final_train[TARGET], test_size=0.15,
                                              stratify=final_train[TARGET], random_state=SEED)
fitted, rows = {}, []
for name in MODELS:
    model = get_model(name).set_params(**{f"clf__{k}": v for k, v in best_params[name].items()}).fit(X_fit, y_fit)
    p = model.predict_proba(X_val)[:, 1]
    thr = best_threshold(y_val, p)
    fitted[name] = (model, thr)
    rows.append({"model": name, "roc_auc": roc_auc_score(y_val, p), "pr_auc": average_precision_score(y_val, p),
                 "f1": f1_score(y_val, p >= thr), "threshold": thr})
tuned = pd.DataFrame(rows).set_index("model").round(4)
display(tuned)

final_name = tuned.roc_auc.idxmax()
final_model, threshold = fitted[final_name]
print(f"FINAL MODEL: {final_name} | decision threshold = {threshold:.3f}")
tuned.to_csv(OUT_DIR / "tuned_model_comparison.csv")

## Final evaluation on the untouched test set

In [ ]:
X_test, y_test = test[FEATURES], test[TARGET].values
p_test = final_model.predict_proba(X_test)[:, 1]
pred = (p_test >= threshold).astype(int)

test_auc, test_ap = roc_auc_score(y_test, p_test), average_precision_score(y_test, p_test)
print(f"TEST  ROC-AUC = {test_auc:.3f} | PR-AUC = {test_ap:.3f} (random baseline = {y_test.mean():.3f})\n")
print(classification_report(y_test, pred, target_names=["On time (<=15 min)", "Delayed (>15 min)"], digits=3))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt=",d", cmap="Blues", ax=ax[0],
            xticklabels=["Pred on-time", "Pred delayed"], yticklabels=["Actual on-time", "Actual delayed"])
ax[0].set_title(f"Confusion matrix (threshold {threshold:.2f})")
fpr, tpr, _ = roc_curve(y_test, p_test); ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "k--", lw=.7)
ax[1].set(title=f"ROC curve (AUC = {test_auc:.3f})", xlabel="False-positive rate", ylabel="True-positive rate")
fr, mp = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
ax[2].plot(mp, fr, marker="o"); ax[2].plot([0, 1], [0, 1], "k--", lw=.7)
ax[2].set(title="Calibration", xlabel="Predicted probability", ylabel="Observed delay rate")
plt.tight_layout(); plt.show()

In [ ]:
# Business view: how well does the model prioritise risky flights? (gains / lift by risk decile)
t = pd.DataFrame({"y": y_test, "p": p_test})
t["risk_decile"] = pd.qcut(t.p.rank(method="first", ascending=False), 10, labels=range(1, 11))
g = t.groupby("risk_decile", observed=True).agg(flights=("y", "size"), delay_rate=("y", "mean"))
g["lift"] = g.delay_rate / y_test.mean()
g["cum_share_of_all_delays_%"] = 100 * (g.flights * g.delay_rate).cumsum() / t.y.sum()
display(g.round(3))
top10, top20 = g.loc[1, "cum_share_of_all_delays_%"], g.loc[2, "cum_share_of_all_delays_%"]
print(f"The 10% riskiest flights contain {top10:.0f}% of all >15-min delays (lift {g.loc[1,'lift']:.1f}x); the riskiest 20% contain {top20:.0f}%.")
g.delay_rate.plot.bar(figsize=(7, 3), rot=0, color="#C44E52"); plt.axhline(y_test.mean(), color="k", ls="--")
plt.title("Observed delay rate by predicted-risk decile (1 = riskiest)"); plt.show()

## What drives delays? (permutation importance by factor group)

In [ ]:
GROUPS = {"Airline": ["AIRLINE_RATE"],
          "Route & airports": ["ORIGIN_RATE", "DEST_RATE", "ROUTE_RATE", "ORIGIN_HOUR_RATE"],
          "Time of day": ["DEP_HOUR", "ARR_HOUR"],
          "Month": ["MONTH"],
          "Day of week / weekend": ["DAY_OF_WEEK", "IS_WEEKEND"],
          "Distance & duration": ["DISTANCE", "SCHEDULED_TIME"]}

Xp = X_test.sample(min(100_000, len(X_test)), random_state=SEED)
yp = test.loc[Xp.index, TARGET]
base_auc = roc_auc_score(yp, final_model.predict_proba(Xp)[:, 1])
rng = np.random.default_rng(SEED)

def auc_drop(cols, repeats=3):
    drops = []
    for _ in range(repeats):
        Xs = Xp.copy()
        Xs[cols] = Xp[cols].values[rng.permutation(len(Xp))]       # shuffle the rows of these columns together
        drops.append(base_auc - roc_auc_score(yp, final_model.predict_proba(Xs)[:, 1]))
    return np.mean(drops)

drivers = pd.DataFrame({"factor": list(GROUPS), "auc_drop": [auc_drop(c) for c in GROUPS.values()]})
drivers = drivers.sort_values("auc_drop", ascending=False).reset_index(drop=True)
drivers["share_%"] = (100 * drivers.auc_drop.clip(lower=0) / drivers.auc_drop.clip(lower=0).sum()).round(1)
features_imp = pd.DataFrame({"feature": FEATURES, "auc_drop": [auc_drop([f]) for f in FEATURES]}).sort_values("auc_drop", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
drivers.iloc[::-1].plot.barh(x="factor", y="auc_drop", legend=False, ax=ax[0], color="#4C72B0"); ax[0].set_title("Delay drivers by factor group")
features_imp.head(10).iloc[::-1].plot.barh(x="feature", y="auc_drop", legend=False, ax=ax[1], color="#55A868"); ax[1].set_title("Top individual features")
plt.tight_layout(); plt.show()
drivers.round(4)

## Export the model bundle

In [ ]:
# Route table (distance, scheduled duration, time-zone shift) so the app can auto-fill route details
r = train[["ROUTE", "DISTANCE", "SCHEDULED_TIME", "DEP_MIN", "ARR_MIN"]].copy()
r["TZ"] = ((r.ARR_MIN - r.DEP_MIN - r.SCHEDULED_TIME + 720) % 1440) - 720     # destination clock - origin clock (minutes)
route_table = r.groupby("ROUTE").agg(distance=("DISTANCE", "median"), sched_time=("SCHEDULED_TIME", "median"),
                                     tz_shift=("TZ", "median"), n=("DISTANCE", "size"))
route_table = route_table[route_table.n >= 20]
route_table["tz_shift"] = (np.round(route_table.tz_shift / 60) * 60).astype(int)

used_airports = set(route_table.index.str.split("_").str[0]) | set(route_table.index.str.split("_").str[1])
bundle = {
    "model": final_model, "features": FEATURES, "threshold": float(threshold), "prior": float(prior),
    "rate_tables": {k: v.to_dict() for k, v in rate_tables.items()},
    "routes": route_table.to_dict(orient="index"),
    "airlines": airline_names[airline_names.index.isin(train.AIRLINE.unique())].to_dict(),
    "airports": {c.IATA_CODE: f"{c.CITY}, {c.STATE} - {c.AIRPORT}" for c in airports.itertuples() if c.IATA_CODE in used_airports},
    "drivers": drivers.to_dict(orient="records"),
    "metrics": {"model": final_name, "roc_auc": test_auc, "pr_auc": test_ap, "base_rate": float(y_test.mean()),
                "precision": float(((pred == 1) & (y_test == 1)).sum() / max(pred.sum(), 1)),
                "recall": float(((pred == 1) & (y_test == 1)).sum() / y_test.sum()),
                "top20_capture_pct": float(top20)},
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, f"{MODEL_DIR}/flight_delay_bundle.joblib", compress=3)
print("Saved:", f"{MODEL_DIR}/flight_delay_bundle.joblib", f"({os.path.getsize(f'{MODEL_DIR}/flight_delay_bundle.joblib')/1e6:.1f} MB)")
print("scikit-learn version used for training:", sklearn.__version__, "(use the same version in the app's requirements.txt)")